# NB04 — Week 6: 1D-CNN Sequence Model (WORKSTATION)
Requires torch + CUDA (RTX 4060 Ti). Cells self-skip if torch is absent so the
notebook suite always executes end-to-end. Conversion target: `aip_signal/cnn_rul.py`.

In [1]:
import importlib.util
import json
from pathlib import Path

import numpy as np
import polars as pl

TORCH = importlib.util.find_spec("torch") is not None
DATA_DIR = Path("../../data")
print("torch available:", TORCH)

torch available: True


## 1. Sliding windows per unit → (N, window, features), target = RUL at window end

In [2]:
meta = json.loads(next((DATA_DIR / "processed").glob("feature_meta_*.json")).read_text())
FD = next((DATA_DIR / "processed").glob("features_*_train.parquet")).stem.split("_")[1]
FEATS, WINDOW = meta["features"], 30
train = pl.read_parquet(DATA_DIR / f"processed/features_{FD}_train.parquet")
test = pl.read_parquet(DATA_DIR / f"processed/features_{FD}_test.parquet")


def make_windows(df, window=WINDOW):
    xs, ys = [], []
    for _, u in df.sort(["unit_id", "cycle"]).group_by("unit_id"):
        arr, rul = u.select(FEATS).to_numpy(), u["rul"].to_numpy()
        if len(arr) < window:
            arr = np.vstack([np.repeat(arr[:1], window - len(arr), axis=0), arr])
            rul = np.concatenate([np.repeat(rul[:1], window - len(rul)), rul])
        for end in range(window, len(arr) + 1):
            xs.append(arr[end - window:end]); ys.append(rul[end - 1])
    return np.stack(xs), np.array(ys, dtype=np.float32)


x_smoke, y_smoke = make_windows(train.filter(pl.col("unit_id") <= 2))
print("window smoke:", x_smoke.shape, y_smoke.shape)  # torch-free shape check

window smoke: (504, 30, 112) (504,)


## 2. Model + training (skips without torch)

In [3]:
if TORCH:
    import mlflow
    import torch
    import torch.nn as nn
    from torch.utils.data import DataLoader, TensorDataset

    dev = "cuda" if torch.cuda.is_available() else "cpu"
    x_tr, y_tr = make_windows(train)
    lastw = test.filter(pl.col("cycle") >= pl.col("cycle").max().over("unit_id") - WINDOW + 1)
    x_te, y_te = make_windows(lastw)

    model = nn.Sequential(
        nn.Conv1d(len(FEATS), 32, 5, padding=2), nn.BatchNorm1d(32), nn.ReLU(),
        nn.Conv1d(32, 64, 5, padding=2), nn.BatchNorm1d(64), nn.ReLU(),
        nn.Conv1d(64, 64, 3, padding=1), nn.BatchNorm1d(64), nn.ReLU(),
        nn.AdaptiveAvgPool1d(1), nn.Flatten(),
        nn.Linear(64, 64), nn.ReLU(), nn.Dropout(0.2), nn.Linear(64, 1)).to(dev)
    opt = torch.optim.AdamW(model.parameters(), 1e-3)
    lossf = nn.SmoothL1Loss()
    dl = DataLoader(TensorDataset(torch.tensor(x_tr).float().permute(0, 2, 1),
                                  torch.tensor(y_tr)), batch_size=256, shuffle=True)
    mlflow.set_tracking_uri(f"sqlite:///{(DATA_DIR / 'processed/mlflow_local.db').resolve()}")
    mlflow.set_experiment("aip/m2/rul-cmapss")
    with mlflow.start_run(run_name=f"cnn1d-{FD}"):
        mlflow.set_tags({"dataset": "cmapss", "fd_set": FD, "model_family": "cnn1d",
                         "feature_version": meta["feature_version"]})
        for ep in range(30):
            for xb, yb in dl:
                xb, yb = xb.to(dev), yb.to(dev)
                opt.zero_grad(); loss = lossf(model(xb).squeeze(-1), yb)
                loss.backward(); opt.step()
        model.eval()
        with torch.no_grad():
            pred = np.clip(model(torch.tensor(x_te).float().permute(0, 2, 1).to(dev))
                           .squeeze(-1).cpu().numpy(), 0, None)
        r = float(np.sqrt(np.mean((y_te - pred) ** 2)))
        mlflow.log_metric("rmse", r)
        print("cnn1d RMSE:", r)
else:
    print("SKIPPED: install torch on the workstation (see runbook) and rerun.")
print("NB04 complete.")

2026/09/19 01:23:56 INFO mlflow.agent.hint: Load the `instrumenting-with-mlflow-tracing` skill at C:\vscode_hywork\dsml_projects\assetintelligenceplatform\aipenv\Lib\site-packages\mlflow\assistant\skills\instrumenting-with-mlflow-tracing\SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


2026/09/19 01:24:02 WARNING mlflow.utils.git_utils: Failed to import Git (the Git executable is probably not on your PATH), so Git SHA is not available. Error: Failed to initialize: Bad git executable.
The git executable must be specified in one of the following ways:
    - be included in your $PATH
    - be set via $GIT_PYTHON_GIT_EXECUTABLE
    - explicitly set via git.refresh(<full-path-to-git-executable>)

All git commands will error until this is rectified.

This initial message can be silenced or aggravated in the future by setting the
$GIT_PYTHON_REFRESH environment variable. Use one of the following values:
    - quiet|q|silence|s|silent|none|n|0: for no message or exception
    - warn|w|warning|log|l|1: for a warning message (logging level CRITICAL, displayed by default)
    - error|e|exception|raise|r|2: for a raised exception

Example:
    export GIT_PYTHON_REFRESH=quiet



2026/09/19 01:24:02 WARNING mlflow.utils.git_utils: Failed to import Git (the Git executable is probably not on your PATH), so Git branch is not available. Error: Failed to initialize: Bad git executable.
The git executable must be specified in one of the following ways:
    - be included in your $PATH
    - be set via $GIT_PYTHON_GIT_EXECUTABLE
    - explicitly set via git.refresh(<full-path-to-git-executable>)

All git commands will error until this is rectified.

This initial message can be silenced or aggravated in the future by setting the
$GIT_PYTHON_REFRESH environment variable. Use one of the following values:
    - quiet|q|silence|s|silent|none|n|0: for no message or exception
    - warn|w|warning|log|l|1: for a warning message (logging level CRITICAL, displayed by default)
    - error|e|exception|raise|r|2: for a raised exception

Example:
    export GIT_PYTHON_REFRESH=quiet



2026/09/19 01:24:02 WARNING mlflow.utils.git_utils: Failed to import Git (the Git executable is probably not on your PATH), so Git repository URL is not available. Error: Failed to initialize: Bad git executable.
The git executable must be specified in one of the following ways:
    - be included in your $PATH
    - be set via $GIT_PYTHON_GIT_EXECUTABLE
    - explicitly set via git.refresh(<full-path-to-git-executable>)

All git commands will error until this is rectified.

This initial message can be silenced or aggravated in the future by setting the
$GIT_PYTHON_REFRESH environment variable. Use one of the following values:
    - quiet|q|silence|s|silent|none|n|0: for no message or exception
    - warn|w|warning|log|l|1: for a warning message (logging level CRITICAL, displayed by default)
    - error|e|exception|raise|r|2: for a raised exception

Example:
    export GIT_PYTHON_REFRESH=quiet



cnn1d RMSE: 8.625797271728516
NB04 complete.
